# 检索对抗攻击分布式拆分 / 合并工具

检索对抗攻击（`run_attack_ultra.py`）逐个优化投毒目标的 `entity.buggy_code`，**单条耗时随 token 序列长度 n 呈 O(n²) 增长**。多机分布式运行时，需要把投毒目标文件拆成 N 份分发到 N 台服务器，跑完后再合并。

## 功能

1. **拆分（Section 1）**：把 `poison_targets_set.json` 按 **计算量 (Σn²)** 均衡拆成 N 个子文件（而非按条数均衡），避免长 token 文档集中导致某台服务器成为长尾。采用 LPT 贪心调度。
2. **合并（Section 2）**：从目录中合并多个子文件的优化结果，可选仅保留已完成（`poisoned_buggy_code` 非空）的条目。
3. **交集（Section 3）**：取多个检索攻击结果的 `(cwe, entity.id)` 公共交集；可通过布尔参数决定生成攻击结果是否也参与交集。开启时从生成文件补入 `generation_attack`，关闭时获得检索侧最大公共交集。

## 输入数据格式（两种，均兼容）

- **Case1**：多个 CWE 键 `{"CWE-20": [entry, ...], "CWE-119": [...], ...}`
- **Case2**：单个 `{"CWE-MIXED": [entry, ...]}`

每个 `entry` 含 `entity.buggy_code`（待优化对象，决定 token 长度）、`retrieval_attack.poisoned_buggy_code`（优化产物，存在且非空 = 已完成）等字段。

> **使用方法**：先运行 Section 0 初始化，再按需运行 Section 1（拆分）或 Section 2（合并）。

## Section 0 — 全局初始化

In [ ]:
# ========== Section 0: 全局初始化 ==========
import os
import sys
import json
import statistics
from pathlib import Path
from collections import defaultdict

# ── 项目根目录上溯 ──
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

from src.utils.io import load_json, save_json, load_yaml


# ── 仅加载 tokenizer 的辅助函数（不加载完整模型权重、不占用 GPU）──
def load_tokenizer(model_config_path: str):
    """从模型配置 yml 读取 model_name，仅加载其 tokenizer。

    Args:
        model_config_path: 模型配置文件路径（如 configs/models/harrier-oss-v1-0.6b.yml）

    Returns:
        transformers.PreTrainedTokenizer 实例（与对应 Embedder 同词表）
    """
    os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
    from transformers import AutoTokenizer

    cfg = load_yaml(str(PROJECT_ROOT / model_config_path)
                    if not os.path.isabs(model_config_path) else model_config_path)
    model_name = cfg["model_name"]
    print(f"加载 tokenizer: {model_name}")
    return AutoTokenizer.from_pretrained(model_name)


print("Section 0 Done — 初始化完成 ✅")

## Section 1 — 按 CWE 分层 + 计算量 (Σn²) 均衡拆分

将投毒目标文件按用户指定的 `target_cwes` 截断后拆成 `num_splits` 个子文件，**在每个目标 CWE 类别内部独立做均衡分配**。已完成检索投毒优化的样本不会再次进入耗时贪心计算，但仍会被确定性分配到某一子文件中，供后续与新完成结果一起合并：

1. **目标 CWE 过滤**：`target_cwes=None` 时保持全部输入 CWE；传入数组时只保留数组中存在于输入文件的 CWE，未知 CWE 或空数组立即报错，避免拼写错误导致静默丢数据。
2. **完成状态分流**：`retrieval_attack.poisoned_buggy_code` 非空的样本视为已完成；其余样本视为待优化。只有待优化样本会执行 tokenizer 编码和 `cost=n²` 计算。
3. **待优化样本 LPT**：逐 CWE 按 `cost` 降序，将样本投入当前累计待优化 `Σn²` 最小且未超过该 CWE 条数配额的分片。
4. **已完成样本补入**：LPT 完成后，逐 CWE 将已完成样本放入该 CWE 当前总条数最少的分片；它们计入文件条数，但不计入 `Σn/Σn²` 待优化负载。
5. **输出结构**：子文件只保留选中的目标 CWE 顶层键（即使某分片下为空）；每个目标样本恰好写入一个子文件，entry 内容不修改。
6. **均衡报告**：同时打印每份总条数、待优化数、已完成数、待优化 `Σn/Σn²` 与耗时占比，并附各 CWE 的总数/待优化/已完成分布。

### Section 1.5 — 分布式攻击配置扇出

拆分完成后，按 `CONFIG_ALIAS_PAIRS = [(别名, 配置路径), ...]`（如 `[("ABGS_Ablation", "configs/attack/retrieval/gte/abgs/abgs.yml"), ("AGGD_Ablation", "configs/attack/retrieval/gte/aggd/aggd.yml")]`）将数组内每个攻击配置复制 `NUM_SPLITS` 份：

1. **输出目录**：与母本配置同级，命名 `<母本目录>/<别名>-{YYYY-M-D}/`（如 `configs/attack/retrieval/gte/abgs/ABGS_Ablation-2026-7-18/`），文件命名 `<stem>{i}.yml`（如 `abgs1.yml` ~ `abgs15.yml`）。
2. **改写 `data.target_file_path`**：解析 `TARGET_FILE_PATH` 得到检索器名（如 `gte`）与文件名（如 `poison_targets_set_v4_tau_2_top_10.json`），将第 `i` 份的 `target_file_path` 行级改写为 `data/query/black/target/{检索器}/split/{文件名stem}_part{i}of{NUM_SPLITS}.json`，与上方拆分输出一一对应。
3. **保留格式**：采用文本级行替换而非 YAML round-trip，母本配置中的注释与排版完整保留。


In [ ]:
# ========== Section 1: 拆分函数定义 ==========
def split_poison_targets(
    target_file_path: str,
    num_splits: int,
    model_config_path: str = "configs/models/harrier-oss-v1-0.6b.yml",
    output_dir: str | None = None,
    target_cwes: list[str] | None = None,
):
    """按目标 CWE 分层，将待优化样本按 cost=n² 均衡拆分并携带已完成样本。

    仅未完成样本参与 tokenizer 与 LPT 计算；已完成样本在 LPT 后按 CWE 条数补入分片，
    因而不会被重复优化，也不会污染待优化计算负载统计。

    Args:
        target_file_path: 投毒目标 JSON 文件路径（{CWE: [entry, ...]} 结构）。
        num_splits: 拆分出的子文件数（即服务器数量）。
        model_config_path: 嵌入模型配置文件路径，用于加载 tokenizer 统计 token 长度。
        output_dir: 子文件输出目录，默认为 <输入文件所在目录>/split。
        target_cwes: 需要保留的顶层 CWE 数组；None 表示保留输入文件全部 CWE。

    Returns:
        子文件路径列表（list[Path]）。
    """
    target_file_path = str(target_file_path)
    if not os.path.isabs(target_file_path):
        target_file_path = str(PROJECT_ROOT / target_file_path)
    in_path = Path(target_file_path)

    data = load_json(target_file_path)
    input_cwe_keys = list(data.keys())
    if target_cwes is None:
        cwe_keys = input_cwe_keys
    else:
        if isinstance(target_cwes, (str, bytes)):
            raise TypeError("target_cwes 必须是 CWE 字符串数组，不能是单个字符串")
        requested = list(target_cwes)
        if not requested:
            raise ValueError("target_cwes 不能为空数组；使用 None 表示保留全部 CWE")
        if any(not isinstance(cwe, str) or not cwe.strip() for cwe in requested):
            raise TypeError("target_cwes 中每一项都必须是非空字符串")
        cwe_keys = list(dict.fromkeys(cwe.strip() for cwe in requested))
        missing_cwes = [cwe for cwe in cwe_keys if cwe not in data]
        if missing_cwes:
            raise KeyError(f"target_cwes 包含输入文件中不存在的 CWE: {missing_cwes}")

    input_total = sum(len(entries or []) for entries in data.values())
    total_entries = sum(len(data.get(cwe, []) or []) for cwe in cwe_keys)
    if total_entries == 0:
        print("[WARN] 目标 CWE 中没有任何投毒目标 entry，未执行拆分。")
        return []

    def _split_is_completed(entry):
        return bool((entry.get("retrieval_attack") or {}).get("poisoned_buggy_code"))

    pending_by_cwe = {
        cwe: [entry for entry in (data.get(cwe, []) or []) if not _split_is_completed(entry)]
        for cwe in cwe_keys
    }
    completed_by_cwe = {
        cwe: [entry for entry in (data.get(cwe, []) or []) if _split_is_completed(entry)]
        for cwe in cwe_keys
    }
    pending_total = sum(len(entries) for entries in pending_by_cwe.values())
    completed_total = sum(len(entries) for entries in completed_by_cwe.values())
    tokenizer = load_tokenizer(model_config_path) if pending_total else None

    num_splits = max(1, int(num_splits))
    if num_splits > total_entries:
        print(f"[WARN] num_splits={num_splits} 超过 entry 总数 {total_entries}，已下调为 {total_entries}。")
        num_splits = total_entries

    # ── 1. 初始化 N 个 bin（仅保留目标 CWE 键结构）──
    bins = [{cwe: [] for cwe in cwe_keys} for _ in range(num_splits)]
    bin_cost = [0] * num_splits            # 待优化样本 Σn²
    bin_tokens = [0] * num_splits          # 待优化样本 Σn
    bin_pending_count = [0] * num_splits   # 待优化条数
    bin_completed_count = [0] * num_splits # 已完成条数
    cwe_pending_dist = {cwe: [0] * num_splits for cwe in cwe_keys}
    cwe_completed_dist = {cwe: [0] * num_splits for cwe in cwe_keys}

    # ── 2. 仅分配待优化样本：CWE 内按 cost 降序 + 带条数配额的 LPT ──
    #    对每个 CWE，将待优化样本按计算量降序逐条投入「当前全局累计 cost 最小、且该 CWE
    #    已分配条数未达配额上限 ceil(N_c/num_splits)」的子文件。条数配额保证每份获得
    #    该 CWE 约 N_c/num_splits 条（±1）；min-cost 选择让全局 Σn² 也尽量均衡。
    import math

    for cwe in cwe_keys:
        items = []  # [(cost, n, entry), ...]
        for entry in pending_by_cwe[cwe]:
            buggy_code = entry.get("entity", {}).get("buggy_code", "") or ""
            n = len(tokenizer.encode(buggy_code, add_special_tokens=True))
            items.append((n * n, n, entry))
        if not items:
            continue

        # CWE 内按计算量降序，大样本优先被均摊到不同 bin
        items.sort(key=lambda x: x[0], reverse=True)

        quota_hi = math.ceil(len(items) / num_splits)  # 每份该 CWE 的条数上限
        cwe_bin_count = [0] * num_splits
        for cost, n, entry in items:
            eligible = [i for i in range(num_splits) if cwe_bin_count[i] < quota_hi]
            idx = min(eligible, key=lambda i: bin_cost[i])
            bins[idx][cwe].append(entry)
            bin_cost[idx] += cost
            bin_tokens[idx] += n
            bin_pending_count[idx] += 1
            cwe_pending_dist[cwe][idx] += 1
            cwe_bin_count[idx] += 1

    # ── 3. 补入已完成样本：不参与 cost，优先放入该 CWE 当前总条数最少的分片 ──
    for cwe in cwe_keys:
        for entry in completed_by_cwe[cwe]:
            idx = min(
                range(num_splits),
                key=lambda i: (
                    cwe_pending_dist[cwe][i] + cwe_completed_dist[cwe][i],
                    bin_pending_count[i] + bin_completed_count[i],
                    i,
                ),
            )
            bins[idx][cwe].append(entry)
            bin_completed_count[idx] += 1
            cwe_completed_dist[cwe][idx] += 1

    expected_entry_refs = sorted(
        id(entry) for cwe in cwe_keys for entry in (data.get(cwe, []) or [])
    )
    assigned_entry_refs = sorted(
        id(entry) for bin_obj in bins for cwe in cwe_keys for entry in bin_obj[cwe]
    )
    if assigned_entry_refs != expected_entry_refs:
        raise AssertionError("拆分结果未满足目标样本恰好分配一次的不变量")

    # ── 4. 写盘 ──
    out_dir = Path(output_dir) if output_dir else in_path.parent / "split"
    if not out_dir.is_absolute():
        out_dir = PROJECT_ROOT / out_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    stem = in_path.stem
    out_paths = []
    for i in range(num_splits):
        out_path = out_dir / f"{stem}_part{i + 1}of{num_splits}.json"
        save_json(bins[i], str(out_path))
        out_paths.append(out_path)

    # ── 5. 均衡报告 ──
    total_cost = sum(bin_cost)
    print(f"\n===== 拆分均衡报告（目标 {total_entries} 个 entry → {num_splits} 份）=====")
    print(f"输入总数={input_total}, 非目标 CWE 过滤={input_total - total_entries}, "
          f"待优化={pending_total}, 已完成并携带={completed_total}")
    print(f"目标 CWE（按输出顺序）: {cwe_keys}")
    print(f"{'子文件':<48}{'总数':>6}{'待优化':>8}{'已完成':>8}"
          f"{'待优化Σn':>12}{'待优化Σn²':>16}{'耗时占比':>10}")
    for i in range(num_splits):
        share = bin_cost[i] / total_cost * 100 if total_cost else 0.0
        total_count = bin_pending_count[i] + bin_completed_count[i]
        print(f"{out_paths[i].name:<48}{total_count:>6}{bin_pending_count[i]:>8}"
              f"{bin_completed_count[i]:>8}{bin_tokens[i]:>12}"
              f"{bin_cost[i]:>16}{share:>9.1f}%")
    if pending_total:
        max_cost, min_cost = max(bin_cost), min(bin_cost)
        imbalance = (max_cost / min_cost) if min_cost > 0 else float("inf")
        print(f"\n最大/最小待优化计算量比 (越接近 1 越均衡): {imbalance:.3f}")
    else:
        print("\n全部目标均已完成优化，无待优化计算负载。")

    # 各 CWE 在各子文件的条数分布（核对分层效果）
    print(f"\n----- 各 CWE 分层分布（单元格=待优化/已完成/总数）-----")
    header = f"{'CWE':<14}{'总数':>6}{'待优化':>8}{'已完成':>8}  " + "".join(
        f"{'part' + str(i + 1):>14}" for i in range(num_splits)
    )
    print(header)
    for cwe in cwe_keys:
        cwe_pending = sum(cwe_pending_dist[cwe])
        cwe_completed = sum(cwe_completed_dist[cwe])
        cwe_total = cwe_pending + cwe_completed
        if cwe_total == 0:
            continue
        cells = []
        for i in range(num_splits):
            p_count = cwe_pending_dist[cwe][i]
            c_count = cwe_completed_dist[cwe][i]
            cells.append(f"{p_count}/{c_count}/{p_count + c_count}")
        row = f"{cwe:<14}{cwe_total:>6}{cwe_pending:>8}{cwe_completed:>8}  " + "".join(
            f"{cell:>14}" for cell in cells
        )
        print(row)
    print(f"\n输出目录: {out_dir}")
    return out_paths

In [ ]:
# ========== Section 1: 执行拆分 ==========
# 投毒目标文件路径（{CWE: [entry, ...]} 结构）
# TARGET_FILE_PATH = "data/query/black/target/gte/poison_targets_set_v4.json"
# TARGET_FILE_PATH = "data/query/black/target/jina/poison_targets_set_v4.json"
TARGET_FILE_PATH = "data/query/black/target/harrier/poison_targets_set_v4.json"

# 仅保留并拆分这些顶层 CWE；None 表示保留输入文件全部 CWE
# TARGET_CWES = ["CWE-125", "CWE-20", "CWE-119", "CWE-787", "CWE-416"]
TARGET_CWES = ["CWE-787", "CWE-416", "CWE-125", "CWE-476", "CWE-20", "CWE-200", "CWE-119", "CWE-362"]


# 拆分出的子文件个数（即服务器数量）
NUM_SPLITS = 10
# 嵌入模型配置（用于加载 tokenizer 统计 token 长度）
# MODEL_CONFIG_PATH = "configs/models/gte-modernbert-base.yml"
# MODEL_CONFIG_PATH = "configs/models/jina-code-embeddings-0.5b.yml"
MODEL_CONFIG_PATH = "configs/models/harrier-oss-v1-0.6b.yml"


# 子文件输出目录，None 表示默认写入 <输入文件所在目录>/split
OUTPUT_DIR = None

split_paths = split_poison_targets(
    target_file_path=TARGET_FILE_PATH,
    num_splits=NUM_SPLITS,
    model_config_path=MODEL_CONFIG_PATH,
    output_dir=OUTPUT_DIR,
    target_cwes=TARGET_CWES,
)

In [ ]:
# ========== Section 1.5: 分布式攻击配置扇出函数定义 ==========
import re
from datetime import datetime


def fanout_attack_configs(
    config_alias_pairs,
    target_file_path: str,
    num_splits: int,
    timestamp: str | None = None,
):
    """将数组内的每个攻击配置文件复制 num_splits 份，并逐份改写 data.target_file_path。

    对每个 (别名, 配置路径) 二元组，在与母本配置同级的目录下创建
    ``<母本目录>/<别名>-{YYYY-M-D}/`` 子目录，将配置复制为 ``<stem>{i}.yml``
    （i = 1..num_splits），并将第 i 份中的 ``data.target_file_path`` 改写为::

        data/query/black/target/{retriever}/split/{stem}_part{i}of{num_splits}.json

    其中 retriever / 文件名解析自 target_file_path，如
    "data/query/black/target/gte/poison_targets_set_v4_tau_2_top_10.json"
    → retriever="gte"，文件名="poison_targets_set_v4_tau_2_top_10.json"。

    采用**文本级行替换**而非 YAML round-trip，完整保留母本配置中的注释与格式。

    Args:
        config_alias_pairs: [(别名, 配置文件路径), ...]，如
            [("ABGS_Ablation", "configs/attack/retrieval/gte/abgs/abgs.yml"),
             ("AGGD_Ablation", "configs/attack/retrieval/gte/aggd/aggd.yml")]。
        target_file_path: 投毒目标文件路径，用于解析检索器名与文件名。
        num_splits: 每个配置复制的份数（须与 split_poison_targets 的拆分份数一致）。
        timestamp: 目录时间戳，默认取当天日期（格式 YYYY-M-D，如 2026-7-18）。

    Returns:
        {别名: [生成的配置文件路径 (Path), ...]}。
    """
    num_splits = max(1, int(num_splits))
    if timestamp is None:
        now = datetime.now()
        timestamp = f"{now.year}-{now.month}-{now.day}"

    # ── 解析 target_file_path：检索器名（父目录名）+ 文件名 stem ──
    tf = Path(str(target_file_path))
    retriever = tf.parent.name  # 如 "gte" / "jina"
    target_stem = tf.stem       # 如 "poison_targets_set_v4_tau_2_top_10"

    result = {}
    for alias, config_path in config_alias_pairs:
        config_path = str(config_path)
        if not os.path.isabs(config_path):
            config_path = str(PROJECT_ROOT / config_path)
        src = Path(config_path)
        if not src.is_file():
            raise FileNotFoundError(f"配置文件不存在: {src}")

        out_dir = src.parent / f"{alias}-{timestamp}"
        out_dir.mkdir(parents=True, exist_ok=True)

        raw = src.read_text(encoding="utf-8")
        out_paths = []
        for i in range(1, num_splits + 1):
            new_target = (
                f"data/query/black/target/{retriever}/split/"
                f"{target_stem}_part{i}of{num_splits}.json"
            )
            # 行级替换 target_file_path 的值，保留注释与其余格式
            new_raw, n_sub = re.subn(
                r'(?m)^(\s*target_file_path\s*:\s*)["\']?[^"\'\n#]+["\']?',
                lambda m: f'{m.group(1)}"{new_target}"',
                raw,
                count=1,
            )
            if n_sub == 0:
                raise ValueError(f"未能在配置中找到 target_file_path 行: {src}")
            out_path = out_dir / f"{src.stem}{i}.yml"
            out_path.write_text(new_raw, encoding="utf-8")
            out_paths.append(out_path)
        result[alias] = out_paths

    # ── 报告 ──
    print(f"\n===== 配置扇出报告（num_splits={num_splits}, 时间戳={timestamp}）=====")
    for alias, paths in result.items():
        print(f"\n[{alias}] 输出目录: {paths[0].parent}")
        for p in paths:
            line = next(
                (ln.strip() for ln in p.read_text(encoding="utf-8").splitlines()
                 if ln.strip().startswith("target_file_path")), ""
            )
            print(f"  - {p.name:<20} {line}")
    return result

In [ ]:
# ========== Section 1.5: 执行配置扇出 ==========
# [(别名, 配置文件路径), ...]：数组内每个配置会被复制 NUM_SPLITS 份

# TARGET_FILE_PATH = "data/query/black/target/gte/poison_targets_set_v4.json"
TARGET_FILE_PATH = "data/query/black/target/jina/poison_targets_set_v4.json"
# TARGET_FILE_PATH = "data/query/black/target/harrier/poison_targets_set_v4.json"

NUM_SPLITS = 10

CONFIG_ALIAS_PAIRS = [
    # ("ABGS-wo-sv", "configs/attack/retrieval/jina/abgs/abgs.yml"),
    # ("ABGS-wo-dl", "configs/attack/retrieval/jina/abgs/abgs.yml"),
    # ("ABGS-wo-dl-sv", "configs/attack/retrieval/jina/abgs/abgs.yml"),
    # ("ABGS-wo-dl-sv-e", "configs/attack/retrieval/jina/abgs/abgs.yml"),
    # ("PABS-patience10","configs/attack/retrieval/jina/pabs/pabs.yml"),
    # ("ABGS-Explore-Naive","configs/attack/retrieval/gte/abgs/abgs.yml"),
    ("ABGS", "configs/attack/retrieval/jina/abgs/abgs.yml"),
    # ("PABS", "configs/attack/retrieval/harrier/pabs/pabs.yml"),
    # ("AGGD", "configs/attack/retrieval/harrier/aggd/aggd.yml"),
    # ("ABGS", "configs/attack/retrieval/gte/abgs/abgs.yml")
]

fanout_results = fanout_attack_configs(
    config_alias_pairs=CONFIG_ALIAS_PAIRS,
    target_file_path=TARGET_FILE_PATH,  # 复用上方 Section 1 的拆分目标
    num_splits=NUM_SPLITS,              # 复用上方 Section 1 的拆分份数
)

## Section 2 — 合并多服务器优化结果

从目录中读取多个子文件并合并优化后的投毒文档：

1. **遍历**：读取 `input_dir` 下所有 `.json`（自动排除已生成的 `*_merged.json`）。
2. **完成判定**：`completed = bool(entry["retrieval_attack"]["poisoned_buggy_code"])`（与 `data_prep_ultra.py` 的 `build_pending_entries` 口径一致）。`only_completed=True` 时仅保留已完成条目。
3. **按 CWE 合并去重**：以 `(cwe, entity.id)` 为键去重，重复时**优先保留已完成**的条目。
4. **输出**：默认写入子文件所在目录，命名 `{公共前缀}_merged.json`；打印每个 CWE 的合并条数与完成率。

In [ ]:
# ========== Section 2: 合并函数定义 ==========
from src.utils.io import normalize_bfp_record_id


def _is_completed(entry: dict) -> bool:
    """优化是否完成：retrieval_attack.poisoned_buggy_code 存在且非空。"""
    ra = entry.get("retrieval_attack", {}) or {}
    return bool(ra.get("poisoned_buggy_code"))


def merge_poison_targets(
    input_dir: str,
    only_completed: bool = True,
    output_dir: str | None = None,
    output_name: str | None = None,
):
    """合并目录下多个子文件中优化后的投毒文档。

    Args:
        input_dir: 子文件所在目录（存放多个待合并的 .json 子文件）。
        only_completed: 是否仅合并已完成优化的数据项（poisoned_buggy_code 非空）。
        output_dir: 合并后文件输出目录，默认为子文件所在目录。
        output_name: 输出文件名（不含目录），默认为 {公共前缀}_merged.json。

    Returns:
        合并后文件路径（Path）。
    """
    input_dir = str(input_dir)
    if not os.path.isabs(input_dir):
        input_dir = str(PROJECT_ROOT / input_dir)
    in_dir = Path(input_dir)
    if not in_dir.is_dir():
        raise NotADirectoryError(f"输入目录不存在: {in_dir}")

    # ── 1. 收集待合并子文件（排除 *_merged.json）──
    sub_files = sorted(
        p for p in in_dir.glob("*.json") if not p.stem.endswith("_merged")
    )
    if not sub_files:
        print(f"[WARN] 目录下未找到可合并的子文件: {in_dir}")
        return None
    print(f"找到 {len(sub_files)} 个子文件待合并:")
    for p in sub_files:
        print(f"  - {p.name}")

    # ── 2. 按 CWE 合并 + (cwe, id) 去重（重复时优先保留已完成）──
    merged: dict[str, list] = defaultdict(list)
    seen: dict[tuple, int] = {}  # (cwe, norm_id) -> index in merged[cwe]
    cwe_order: list[str] = []    # 保留首次出现的 CWE 键顺序
    skipped_incomplete = 0

    for sf in sub_files:
        data = load_json(str(sf))
        for cwe, entries in data.items():
            if cwe not in cwe_order:
                cwe_order.append(cwe)
            for entry in entries or []:
                completed = _is_completed(entry)
                if only_completed and not completed:
                    skipped_incomplete += 1
                    continue
                ent_id = entry.get("entity", {}).get("id")
                key = (cwe, normalize_bfp_record_id(ent_id))
                if key in seen:
                    # 已存在：仅当旧条目未完成、新条目已完成时替换
                    idx = seen[key]
                    if completed and not _is_completed(merged[cwe][idx]):
                        merged[cwe][idx] = entry
                else:
                    seen[key] = len(merged[cwe])
                    merged[cwe].append(entry)

    # 按首次出现顺序整理输出 dict
    merged_out = {cwe: merged.get(cwe, []) for cwe in cwe_order}

    # ── 3. 写盘 ──
    out_dir = Path(output_dir) if output_dir else in_dir
    if not out_dir.is_absolute():
        out_dir = PROJECT_ROOT / out_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    if output_name:
        out_name = output_name if output_name.endswith(".json") else f"{output_name}.json"
    else:
        prefix = os.path.commonprefix([p.stem for p in sub_files]).rstrip("_-")
        # 去除尾部形如 "_part" 的拆分标记残留
        for suffix in ("_part", "part"):
            if prefix.endswith(suffix):
                prefix = prefix[: -len(suffix)].rstrip("_-")
                break
        out_name = f"{(prefix or 'poison_targets_set')}_merged.json"
    out_path = out_dir / out_name
    save_json(merged_out, str(out_path))

    # ── 4. 合并报告 ──
    print(f"\n===== 合并报告（only_completed={only_completed}）=====")
    total = 0
    for cwe in cwe_order:
        entries = merged_out[cwe]
        done = sum(1 for e in entries if _is_completed(e))
        total += len(entries)
        rate = (done / len(entries) * 100) if entries else 0.0
        print(f"  {cwe:<14} 条数={len(entries):>5}  已完成={done:>5} ({rate:.1f}%)")
    print(f"  {'合计':<14} 条数={total:>5}")
    if only_completed and skipped_incomplete:
        print(f"  （已跳过未完成条目 {skipped_incomplete} 条）")
    print(f"\n输出文件: {out_path}")
    return out_path

In [ ]:
# ========== Section 2: 执行合并 ==========
# 子文件所在目录（各服务器跑完后回收的子文件存放处）
INPUT_DIR = "data/query/black/target/jina/retrieval_attack/Ablation/abgs_wo_sv/split"
# 是否仅合并已完成优化的数据项（poisoned_buggy_code 非空）
ONLY_COMPLETED = True
# 合并后文件输出目录，None 表示默认写入子文件所在目录
OUTPUT_DIR = None

merged_path = merge_poison_targets(
    input_dir=INPUT_DIR,
    only_completed=ONLY_COMPLETED,
    output_dir=OUTPUT_DIR,
)

## Section 3 — 多个检索攻击结果公共交集（可选生成侧）+ 按预算截断

接收 **n 个**检索攻击结果文件（对应 n 种检索对抗算法，如 ABGS/AGGD/PABS/AGGS）以及可选的**单个**生成攻击结果文件，按以下**三段式**流程写出 n 份投毒文件。布尔参数 `include_generation_in_intersection` 控制生成侧是否参与身份键交集，默认为 `True` 以保持原行为：

1. **逐文件 × 逐 CWE 构造候选集**：当 `include_generation_in_intersection=True` 时，每个检索文件分别与生成文件按 `(cwe, entity.id)` 取交集，并从生成文件补入已完成的 `generation_attack`；为 `False` 时完全不读取生成文件，候选集就是该检索文件中已完成的条目，并保留检索文件原有的 `generation_attack`（若存在）。
2. **逐 CWE 公共 id 交集**：对上一步得到的 n 份候选结果逐 CWE 取**公共 id 交集**——某 `(cwe, entity.id)` 必须在**所有** n 份检索文件中出现才保留；任一检索文件缺失即从全体移除。此步后 n 份文件的 `(cwe, entity.id)` 键集合完全一致。
3. **一次性预算截断**：仅计算**一套** `kept_ids`（逐 CWE 保留的 id 集合）并套用到全部 n 份文件，最终各文件目标集合逐 CWE 严格相同。

> **动机（公平性）**：不同检索对抗算法各自"已完成"的样本集合不同，若各自独立取交集 + 截断，会导致保留的投毒目标集合彼此偏离，使 ASR-R 评估不公。通过公共 id 交集 + 共享截断方案，保证所有算法在**完全相同**的投毒目标上被评估；各文件仅 `poisoned_buggy_code` 内容不同（即各算法自身的对抗优化差异，正是要保留的）。

### 完成判定口径

- **检索攻击已完成**：`entry["retrieval_attack"]["poisoned_buggy_code"]` 存在且非空
- **生成攻击已完成**：仅当 `include_generation_in_intersection=True` 时要求 `entry["generation_attack"]["jailbreak_attack"]["jailbreak_code"]` 存在且非空

### 组装规则

以 `(cwe, entity.id)` 为匹配键；是否要求生成侧匹配由 `include_generation_in_intersection` 决定：

- `entity`：取自检索攻击文件（各文件相同，以检索文件为主源）
- `retrieval_attack`：**整体**取自检索攻击文件（含 `poisoned_buggy_code`、`configs`、`positive_samples`、`negative_samples`）
- `generation_attack`：开关为 `True` 时整体取自生成攻击文件（含 `vinj_attack`、`jailbreak_attack`）；开关为 `False` 时保留检索文件中的原值（若存在），且不保证越狱生成已完成

输出文件保留 CWE 键顺序（按**首个**检索文件中首次出现的顺序），空 CWE 桶不写入。默认输出路径为各检索文件同目录下 `{stem}_intersected.json`。

> **注意**：关闭生成侧交集用于获得多个检索数据集的最大公共交集。如果检索文件自身没有已完成的 `jailbreak_code`，此输出不能直接用于依赖越狱代码作为 poisoned `fixed_code` 的后续评估；应先补齐生成攻击结果，或重新以开关为 `True` 执行合并。

### 按投毒预算截断

由于漏洞注入成功率约 90%，划分投毒目标时会取略高于目标投毒率的规模（超额供给），因此公共交集后需按**期望投毒预算**截断到目标规模。相关参数：

- `corpus_size`：APR 语料库规模（默认 `50000`）
- `poison_rate`：投毒率（默认 `0.005`，即 0.5%）
- `initial_target_file_path`：初始（超额）投毒目标文件路径，仅用于读取各 CWE 的初始条数以计算分配比例；为 `None` 时跳过截断，保留全部公共交集

截断规则：

1. **预算**：`budget = round(corpus_size × poison_rate)`（如 `50000 × 0.5% = 250`）。
2. **公共交集规模 < 预算时不截断**：若 `budget >= 公共交集总数`（或未提供初始文件），直接保留全部公共交集，各文件原样写出。
3. **保持初始 CWE 比例**：以初始文件中各 CWE 条数占比为权重，用**最大余数法（Hamilton）**把 `budget` 分配到各 CWE；当某 CWE 公共交集条数不足以承接其配额时封顶，并把剩余预算按同比例重分配给仍有余量的 CWE，尽量凑齐预算（比例保持“近似”）。
4. **按优化顺序保留（共享 `kept_ids`）**：每个 CWE 内按 `entity.buggy_code` 长度升序（廉价优先，与 `data_prep_ultra.py` 的 `build_pending_entries` 调度口径一致，与运行无关且确定），保留前「分配数」个 id 作为共享的 `kept_ids`。由于该集合对全部 n 份文件统一套用，保证跨算法保留的投毒样本完全一致，从而保证评估公平性。

In [ ]:
# ========== Section 3: 交集函数定义 ==========
import math

from src.utils.io import (
    apply_cross_cwe_id_map, build_cross_cwe_id_map, identity_snapshot,
    normalize_bfp_record_id, validate_global_entity_ids,
)


def _is_retrieval_done(entry: dict) -> bool:
    """检索攻击是否完成：retrieval_attack.poisoned_buggy_code 存在且非空。"""
    return bool((entry.get("retrieval_attack") or {}).get("poisoned_buggy_code"))


def _is_generation_done(entry: dict) -> bool:
    """生成攻击是否完成：generation_attack.jailbreak_attack.jailbreak_code 存在且非空。"""
    ga = entry.get("generation_attack") or {}
    ja = ga.get("jailbreak_attack") or {}
    return bool(ja.get("jailbreak_code"))


def _allocate_budget_by_ratio(
    budget: int,
    ratio_counts: dict,
    avail_counts: dict,
) -> dict:
    """按初始比例（最大余数法）在各 CWE 间分配投毒预算，并受各 CWE 可用交集条数上限约束。

    当某些 CWE 因可用交集条数不足被“封顶”导致总量未达预算时，把剩余预算按同一
    比例重新分配给仍有余量的 CWE（再次最大余数法），迭代直至预算用尽或无余量可分。

    Args:
        budget: 总投毒预算（poison_rate * corpus_size）。
        ratio_counts: {cwe: 初始投毒目标文件中该 CWE 的条数}，用于计算分配比例。
        avail_counts: {cwe: 该 CWE 交集中可用的条数}，作为分配上限。

    Returns:
        {cwe: 分配到的保留条数}（各值 <= avail_counts[cwe]，总和为
        min(budget, sum(avail_counts.values()))）。
    """
    cwes = list(avail_counts.keys())
    alloc = {cwe: 0 for cwe in cwes}
    total_avail = sum(avail_counts.values())
    remaining = min(int(budget), total_avail)

    while remaining > 0:
        # 本轮参与分配的 CWE（仍有剩余可用条数）
        active = [c for c in cwes if avail_counts[c] - alloc[c] > 0]
        if not active:
            break
        weight = {c: ratio_counts.get(c, 0) for c in active}
        total_weight = sum(weight.values())
        if total_weight <= 0:
            # 初始文件对这些 CWE 均无计数：退化为按剩余可用条数分配
            weight = {c: (avail_counts[c] - alloc[c]) for c in active}
            total_weight = sum(weight.values())
            if total_weight <= 0:
                break

        # 最大余数法：先按 floor 分配，再把余数按小数部分从大到小 +1
        raw = {c: remaining * weight[c] / total_weight for c in active}
        floor = {c: int(math.floor(raw[c])) for c in active}
        add = {c: min(floor[c], avail_counts[c] - alloc[c]) for c in active}
        for c in active:
            alloc[c] += add[c]
        given = sum(add.values())
        remaining -= given

        if remaining > 0:
            frac = sorted(
                active,
                key=lambda c: (raw[c] - floor[c], ratio_counts.get(c, 0)),
                reverse=True,
            )
            progressed = False
            for c in frac:
                if remaining <= 0:
                    break
                if avail_counts[c] - alloc[c] > 0:
                    alloc[c] += 1
                    remaining -= 1
                    progressed = True
            # 全部封顶且本轮 floor 也未推进：退出避免死循环
            if not progressed and given == 0:
                break

    return alloc


def _intersect_single(
    ret_data: dict,
    generation_idx: dict,
    cwe_order: list,
    include_generation_in_intersection: bool,
) -> tuple:
    """构造单个检索结果的已完成候选集，并按开关决定是否与生成侧取交集。

    entity / retrieval_attack 始终取自检索文件。开关开启时，仅保留生成侧已完成的
    同身份条目并使用其 generation_attack；关闭时不按生成侧过滤，保留检索条目中已有的
    generation_attack（若存在）。键为 (cwe, norm_id)，同一 id 在不同 CWE 下各自保留。

    Args:
        ret_data: 单个检索攻击结果文件内容（{cwe: [entry, ...]}）。
        generation_idx: 生成攻击索引 {(cwe, norm_id): gen_entry}（仅已完成）；关闭开关时为空。
        cwe_order: 输出 CWE 键顺序（以首个检索文件为准）。
        include_generation_in_intersection: 是否要求条目同时存在于已完成的生成攻击结果中。

    Returns:
        (intersect, done_by_cwe, ret_total, ret_done)
        - intersect: {cwe: [combined_entry]}；生成侧是否参与筛选由开关决定。
        - done_by_cwe: {cwe: 该 CWE 检索已完成条数}，用于报告。
        - ret_total / ret_done: 检索文件的总条数 / 已完成条数。
    """
    retrieval_idx: dict = {}
    done_by_cwe: dict = {}
    ret_total = 0
    ret_done = 0
    for cwe, entries in ret_data.items():
        for entry in entries or []:
            ret_total += 1
            if not _is_retrieval_done(entry):
                continue
            ret_done += 1
            done_by_cwe[cwe] = done_by_cwe.get(cwe, 0) + 1
            norm_id = normalize_bfp_record_id(entry.get("entity", {}).get("id"))
            retrieval_idx[(cwe, norm_id)] = entry

    intersect: dict = {}
    for cwe in cwe_order:
        combined = []
        for (k_cwe, k_id), ret_entry in retrieval_idx.items():
            if k_cwe != cwe:
                continue
            combined_entry = {
                "entity"          : ret_entry["entity"],
                "retrieval_attack": ret_entry["retrieval_attack"],
            }
            if include_generation_in_intersection:
                gen_entry = generation_idx.get((k_cwe, k_id))
                if gen_entry is None:
                    continue
                combined_entry["generation_attack"] = gen_entry["generation_attack"]
            elif "generation_attack" in ret_entry:
                combined_entry["generation_attack"] = ret_entry["generation_attack"]
            combined.append(combined_entry)
        if combined:
            intersect[cwe] = combined
    return intersect, done_by_cwe, ret_total, ret_done


def intersect_attack_results(
    retrieval_file_paths,
    generation_file_path: str | None,
    output_paths=None,
    corpus_size: int = 50000,
    poison_rate: float = 0.005,
    initial_target_file_path: str | None = None,
    include_generation_in_intersection: bool = True,
) -> list:
    """取 n 个检索攻击结果的逐 CWE 公共 id 交集，可选让生成攻击结果参与交集，
    再用同一套预算截断方案写出 n 份投毒文件。

    这样不同检索对抗算法（ABGS/AGGD/PABS/AGGS）最终会在完全相同的 (cwe, entity.id)
    投毒目标集合上被评估，消除 ASR-R 评估不公；各文件仅 poisoned_buggy_code 内容不同
    （即各算法自身的对抗优化结果，正是需要保留的差异）。

    Args:
        retrieval_file_paths: 检索攻击结果文件路径，str 或 list[str]（含 poisoned_buggy_code）。
        generation_file_path: 生成攻击结果文件路径（含 jailbreak_code）；关闭生成侧交集时
                      可为 None，且即使传入也不会读取。
        output_paths: 输出路径列表，需与 retrieval_file_paths 等长；None 时各文件默认写入
                      检索文件同目录下 {retrieval_stem}_intersected.json。
        corpus_size: APR 语料库规模，用于计算投毒预算（默认 50000）。
        poison_rate: 投毒率，用于计算投毒预算（默认 0.005，即 0.5%）。
        initial_target_file_path: 初始（超额）投毒目标文件路径，仅用于读取各 CWE 的初始条数
                      以计算截断分配比例；为 None 时跳过截断，保留全部公共交集。
        include_generation_in_intersection: True 时保持原行为，要求检索条目同时存在于已完成的
                      生成攻击结果；False 时只取多个检索结果的最大公共交集。

    Returns:
        输出文件路径列表（list[Path]），顺序与 retrieval_file_paths 对应。
    """
    # ── 路径规范化 ──
    def _abs(p: str) -> str:
        return p if os.path.isabs(p) else str(PROJECT_ROOT / p)

    # ── 0. 入参规范化 ──
    if isinstance(retrieval_file_paths, (str, os.PathLike)):
        retrieval_file_paths = [retrieval_file_paths]
    retrieval_file_paths = list(retrieval_file_paths)
    if not retrieval_file_paths:
        raise ValueError("retrieval_file_paths 不能为空")
    if not isinstance(include_generation_in_intersection, bool):
        raise TypeError("include_generation_in_intersection 必须是 bool")
    ret_paths = [Path(_abs(str(p))) for p in retrieval_file_paths]

    if output_paths is not None:
        if len(output_paths) != len(ret_paths):
            raise ValueError(
                f"output_paths 长度({len(output_paths)}) 与 retrieval_file_paths "
                f"长度({len(ret_paths)}) 不一致"
            )
        out_paths = [Path(_abs(str(p))) for p in output_paths]
    else:
        out_paths = [rp.parent / f"{rp.stem}_intersected.json" for rp in ret_paths]

    # ── 1. 按开关决定是否加载生成侧并构建已完成索引 ──
    generation_idx: dict = {}
    gen_total = 0
    gen_done = 0
    gen_path = None
    if include_generation_in_intersection:
        if not generation_file_path:
            raise ValueError(
                "include_generation_in_intersection=True 时必须提供 generation_file_path"
            )
        gen_path = Path(_abs(str(generation_file_path)))
        gen_data = load_json(str(gen_path))
        for cwe, entries in gen_data.items():
            for entry in entries or []:
                gen_total += 1
                if not _is_generation_done(entry):
                    continue
                gen_done += 1
                norm_id = normalize_bfp_record_id(entry.get("entity", {}).get("id"))
                generation_idx[(cwe, norm_id)] = entry

    # ── 2. 构造每个检索文件的候选集；可选与生成攻击取交集 ──
    ret_data_list = [load_json(str(rp)) for rp in ret_paths]
    cwe_order = list(ret_data_list[0].keys())  # 以首个检索文件的键顺序为输出基准

    per_file_intersect: list = []
    per_file_done_by_cwe: list = []
    per_file_totals: list = []  # [(ret_total, ret_done), ...]
    for ret_data in ret_data_list:
        inter, done_by_cwe, ret_total, ret_done = _intersect_single(
            ret_data, generation_idx, cwe_order, include_generation_in_intersection
        )
        per_file_intersect.append(inter)
        per_file_done_by_cwe.append(done_by_cwe)
        per_file_totals.append((ret_total, ret_done))

    # ── 3. 逐 CWE 公共 id 交集（跨全部 n 份文件）──
    # 某 id 必须在所有文件的该 CWE 交集中都出现才保留；任一文件缺失即从全体移除。
    common_ids: dict = {}
    entity_by_key: dict = {}  # (cwe, norm_id) -> entity（取自文件 0，供排序用）
    for cwe in cwe_order:
        id_sets = []
        for inter in per_file_intersect:
            ids = {
                normalize_bfp_record_id(e.get("entity", {}).get("id"))
                for e in inter.get(cwe, [])
            }
            id_sets.append(ids)
        common = set.intersection(*id_sets) if id_sets else set()
        if common:
            common_ids[cwe] = common
            for e in per_file_intersect[0].get(cwe, []):
                nid = normalize_bfp_record_id(e.get("entity", {}).get("id"))
                if nid in common and (cwe, nid) not in entity_by_key:
                    entity_by_key[(cwe, nid)] = e.get("entity", {})

    # ── 4. 预算截断（一次性计算 kept_ids，跨文件一致）──
    budget = int(round(corpus_size * poison_rate))
    avail_counts = {cwe: len(ids) for cwe, ids in common_ids.items()}
    total_common = sum(avail_counts.values())

    truncation_applied = False
    alloc: dict = {}
    initial_counts: dict = {}
    kept_ids: dict = {}  # {cwe: set(norm_id)}

    if not initial_target_file_path:
        print("[INFO] 未提供 initial_target_file_path，跳过预算截断，保留全部公共交集。")
        kept_ids = {cwe: set(ids) for cwe, ids in common_ids.items()}
    elif total_common == 0:
        print("[WARN] 公共交集为空，跳过预算截断。")
        kept_ids = {}
    elif budget >= total_common:
        print(
            f"[WARN] 预算 budget={budget} >= 公共交集总数 {total_common}，"
            "无需截断，保留全部公共交集。"
        )
        kept_ids = {cwe: set(ids) for cwe, ids in common_ids.items()}
    else:
        init_path = Path(_abs(str(initial_target_file_path)))
        init_data = load_json(str(init_path))
        inter_cwes = [c for c in cwe_order if avail_counts.get(c, 0) > 0]
        initial_counts = {c: len(init_data.get(c, []) or []) for c in inter_cwes}
        missing = [c for c in inter_cwes if initial_counts.get(c, 0) == 0]
        if missing:
            print(f"[WARN] 初始文件缺少以下 CWE 的计数，其分配比例按 0 处理: {missing}")
        alloc = _allocate_budget_by_ratio(budget, initial_counts, avail_counts)
        for cwe in inter_cwes:
            keep_n = alloc.get(cwe, 0)
            if keep_n <= 0:
                continue
            # 廉价优先：buggy_code 长度升序（与优化调度一致，运行无关且确定），
            # tie-break 归一化 id；entity 取自公共交集（各文件同一 entity），确保各文件
            # 选出的 id 完全一致
            ordered_ids = sorted(
                common_ids[cwe],
                key=lambda nid: (
                    len(entity_by_key.get((cwe, nid), {}).get("buggy_code", "") or ""),
                    str(nid),
                ),
            )
            kept_ids[cwe] = set(ordered_ids[:keep_n])
        truncation_applied = True

    # ── 5. 套用 kept_ids 到每个文件并写出 ──
    def _sort_key(e):
        nid = normalize_bfp_record_id(e.get("entity", {}).get("id"))
        return (
            len(e.get("entity", {}).get("buggy_code", "") or ""),
            str(nid),
        )

    # 先构造全部输出，再用同一套 (CWE, original_id) 映射处理跨 CWE 重复 ID。
    prepared_outputs: list = []  # [(out_path, out_obj, kept_count)]
    for inter, out_path in zip(per_file_intersect, out_paths):
        out_obj: dict = {}
        cnt = 0
        for cwe in cwe_order:
            keep = kept_ids.get(cwe)
            if not keep:
                continue
            selected = [
                e for e in inter.get(cwe, [])
                if normalize_bfp_record_id(e.get("entity", {}).get("id")) in keep
            ]
            if not selected:
                continue
            selected.sort(key=_sort_key)
            out_obj[cwe] = selected
            cnt += len(selected)
        prepared_outputs.append((out_path, out_obj, cnt))

    id_mapping = (
        build_cross_cwe_id_map(prepared_outputs[0][1]) if prepared_outputs else {}
    )
    validated_outputs: list = []  # [(out_path, out_obj, kept_count, renamed)]
    expected_snapshot = None
    for out_path, out_obj, cnt in prepared_outputs:
        renamed = 0
        for cwe, records in out_obj.items():
            renamed += apply_cross_cwe_id_map(records, cwe, id_mapping)
        validate_global_entity_ids(out_obj)
        snapshot = identity_snapshot(out_obj)
        if expected_snapshot is None:
            expected_snapshot = snapshot
        elif snapshot != expected_snapshot:
            raise AssertionError("各优化器输出的 (CWE, original_id) -> entity.id 映射不一致")
        validated_outputs.append((out_path, out_obj, cnt, renamed))

    # 所有输出都通过全局唯一性和映射一致性断言后，再开始写盘。
    written: list = []  # [(out_path, kept_count)]
    for out_path, out_obj, cnt, renamed in validated_outputs:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        save_json(out_obj, str(out_path))
        written.append((out_path, cnt))
        print(f"  [ID] {out_path.name}: 重命名 {renamed} 条跨 CWE 冲突样本")

    # ── 6. 报告 ──
    print(f"\n===== 交集合并报告（{len(ret_paths)} 个检索文件）=====")
    if include_generation_in_intersection:
        print(
            f"  生成侧参与交集: 是；文件={gen_path.name} "
            f"(总条数={gen_total}, 已完成={gen_done})"
        )
        candidate_label = "与生成交集"
    else:
        print("  生成侧参与交集: 否（generation_file_path 未读取）")
        candidate_label = "检索候选"
    for idx, rp in enumerate(ret_paths):
        r_total, r_done = per_file_totals[idx]
        inter = per_file_intersect[idx]
        inter_cnt = sum(len(v) for v in inter.values())
        print(
            f"\n  [文件{idx+1}] {rp.name}"
            f"  (总条数={r_total}, 已完成={r_done}, {candidate_label}={inter_cnt})"
        )
        print(f"    {'CWE':<14}{'检索已完成':>10}{candidate_label:>12}")
        print(f"    {'-'*36}")
        done_by_cwe = per_file_done_by_cwe[idx]
        for cwe in cwe_order:
            r_c = done_by_cwe.get(cwe, 0)
            i_c = len(inter.get(cwe, []))
            print(f"    {cwe:<14}{r_c:>10}{i_c:>12}")

    # ── 6.5 公共交集 / 预算截断报告 ──
    total_initial = sum(initial_counts.values()) or 1
    print(f"\n===== 公共交集 / 预算截断报告 =====")
    print(
        f"  语料库规模={corpus_size}, 投毒率={poison_rate:.4%}, 预算 budget={budget}, "
        f"公共交集总数={total_common}"
    )
    if truncation_applied:
        print(f"  初始文件: {Path(_abs(str(initial_target_file_path))).name}")
        print(
            f"  {'CWE':<14}{'初始数':>8}{'初始占比':>10}{'公共交集':>10}{'分配':>8}{'保留':>8}"
        )
        print(f"  {'-'*58}")
    else:
        print(f"  （未截断，保留全部公共交集）")
        print(f"  {'CWE':<14}{'公共交集':>10}{'保留':>8}")
        print(f"  {'-'*32}")
    kept_total = 0
    for cwe in cwe_order:
        c_common = avail_counts.get(cwe, 0)
        c_keep = len(kept_ids.get(cwe, set()))
        if c_common == 0 and c_keep == 0:
            continue
        kept_total += c_keep
        if truncation_applied:
            n_init = initial_counts.get(cwe, 0)
            share = n_init / total_initial * 100
            a_c = alloc.get(cwe, 0)
            print(
                f"  {cwe:<14}{n_init:>8}{share:>9.1f}%{c_common:>10}{a_c:>8}{c_keep:>8}"
            )
        else:
            print(f"  {cwe:<14}{c_common:>10}{c_keep:>8}")
    if truncation_applied:
        print(
            f"  {'合计':<14}{sum(initial_counts.values()):>8}{'':>10}"
            f"{total_common:>10}{sum(alloc.values()):>8}{kept_total:>8}"
        )
        if kept_total < budget:
            print(f"  [WARN] 公共交集样本不足，实际保留 {kept_total} < 预算 {budget}。")
    else:
        print(f"  {'合计':<14}{total_common:>10}{kept_total:>8}")

    # ── 6.6 输出文件（各文件保留 (cwe,id) 目标集合一致，条数应相等）──
    print(f"\n===== 输出文件 =====")
    for idx, (out_path, cnt) in enumerate(written):
        print(f"  [文件{idx+1}] {out_path}  (保留 {cnt} 条)")

    return [p for p, _ in written]

In [ ]:
# ========== Section 3: 执行交集合并 + 预算截断 ==========
# 检索攻击结果文件路径列表（含 poisoned_buggy_code），可同时传入 n 个不同检索对抗
# 算法（ABGS/AGGD/PABS/AGGS）的结果文件；脚本会先构造各检索文件的已完成候选集，
# 再对 n 份结果取逐 CWE 公共 id 交集，最后用同一套截断方案写出。


# RETRIEVAL_FILE_PATHS = [
#     "data/query/black/target/harrier/retrieval_attack/ABGS-wo-E/poison_targets_set_v4_merged.json",
#     "data/query/black/target/harrier/retrieval_attack/ABGS/poison_targets_set_v4_merged.json",
# ]

RETRIEVAL_FILE_PATHS = [
    "data/query/black/target/harrier/retrieval_attack/Naive/poison_targets_set_v4.json",
    "data/query/black/target/harrier/retrieval_attack/ABGS-wo-E/poison_targets_set_v4_merged.json",
    "data/query/black/target/harrier/retrieval_attack/AGGD/poison_targets_set_v4_merged.json",
    "data/query/black/target/harrier/retrieval_attack/PABS/poison_targets_set_v4_merged.json",
]

# RETRIEVAL_FILE_PATHS = [
#     "data/query/black/target/jina/retrieval_attack/Ablation/abgs/poison_targets_set_v4_tau_2_top_10_merged.json",
#     "data/query/black/target/jina/retrieval_attack/Ablation/abgs_beam3_wo_dl_sv/poison_targets_set_v4_tau_2_top_10_merged.json",
#     "data/query/black/target/jina/retrieval_attack/Ablation/abgs_wo_dl/poison_targets_set_v4_tau_2_top_10_merged.json",
#     "data/query/black/target/jina/retrieval_attack/Ablation/abgs_wo_sv/poison_targets_set_v4_tau_2_top_10_merged.json",
#     "data/query/black/target/jina/retrieval_attack/Ablation/abgs_wo_dl_sv/poison_targets_set_v4_tau_2_top_10_merged.json",
# ]


# 是否让生成攻击结果参与交集：True 保持原行为；False 只取多个检索侧数据集的最大公共交集
INCLUDE_GENERATION_IN_INTERSECTION = True
# INCLUDE_GENERATION_IN_INTERSECTION = False

# 生成攻击结果文件路径（含 jailbreak_code）；关闭上述开关时不会读取，可设为 None
# GENERATION_FILE_PATH = "data/query/black/target/gte/generation_attack/poison_targets_set_v4.json"
GENERATION_FILE_PATH = "data/query/black/target/harrier/generation_attack/poison_targets_set_v4.json"
# GENERATION_FILE_PATH = "data/query/black/target/jina/generation_attack/poison_targets_set_v4.json"


# 输出路径列表，需与 RETRIEVAL_FILE_PATHS 等长；None 表示各文件默认写入检索文件
# 同目录下 {stem}_intersected.json（即各算法目录下各自产出）
OUTPUT_PATHS = None

# ── 预算截断参数 ──
# APR 语料库规模
CORPUS_SIZE = 50000
# 投毒率（默认 0.5%）
POISON_RATE = 0.0065
# 初始（超额）投毒目标文件路径，用于按其各 CWE 条数比例分配预算；None 表示不截断
# INITIAL_TARGET_FILE_PATH = "data/query/black/target/gte/poison_targets_set_v4.json"
INITIAL_TARGET_FILE_PATH = "data/query/black/target/harrier/poison_targets_set_v4.json"
# INITIAL_TARGET_FILE_PATH = "data/query/black/target/jina/poison_targets_set_v4.json"


intersected_paths = intersect_attack_results(
    retrieval_file_paths=RETRIEVAL_FILE_PATHS,
    generation_file_path=GENERATION_FILE_PATH,
    output_paths=OUTPUT_PATHS,
    corpus_size=CORPUS_SIZE,
    poison_rate=POISON_RATE,
    initial_target_file_path=INITIAL_TARGET_FILE_PATH,
    include_generation_in_intersection=INCLUDE_GENERATION_IN_INTERSECTION,
)